# 02807 Computational Tools for Data Science: AVA Data Cleaning & Feature Extraction

This notebook implements the data ingestion, cleaning, structural feature calculation, and CLIP embedding extraction pipeline for the **AVA Aesthetic Dataset** (`trojblue/AVA-Huggingface`).

### Why We Extract Both Feature Types:
1. **CLIP Embeddings (512-dim):** Captures high-level semantic meaning (*what* is in the photo: subject matter, scene type, mood).
2. **Structural Features (12-dim):** Captures low-level photographic execution (*how* the photo is shot: luminance contrast, brightness, center-to-edge sharpness ratio / bokeh proxy, RGB color distributions, and aspect ratio). These low-dimensional features are essential for our **Course Topic 1 (General Clustering & Davies-Bouldin Index)** and bridge AVA's visual patterns with Unsplash's EXIF camera metadata.

## Step 1: Install & Import Dependencies
Run the installation cell below if you do not have `datasets`, `transformers`, or `pyarrow` installed in your environment.

In [ ]:
%pip install datasets transformers torch pillow numpy pyarrow huggingface_hub tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%hf auth login

UsageError: Line magic function `%hf` not found.


In [ ]:
# !pip install -q datasets transformers torch pillow numpy pyarrow huggingface_hub

import numpy as np
import torch
from PIL import Image, ImageChops
from datasets import load_dataset, Dataset
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

/Users/emilsvigants/miniforge3-dtu/envs/dtu02452/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 13:31:15.777000 77418 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Step 2: Configure Hardware & Load Frozen CLIP Backbone
This automatically selects Apple Silicon GPU (`mps`), NVIDIA GPU (`cuda`), or CPU, and loads OpenAI's `clip-vit-base-patch32` in evaluation mode so we don't compute gradients.

In [ ]:
device = (
    "mps" if torch.backends.mps.is_available() 
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"Using device: {device}")

model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_id)

Using device: mps


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 44705.10it/s]


## Step 3: Define Image Cleaning & Structural Feature Functions
- **`trim_uniform_borders`**: Many AVA images come from DPChallenge photography contests and have artificial solid black or white borders added by users. We strip these borders so they don't corrupt color histograms or contrast calculations.
- **`extract_structural_features`**: Computes 9 core pixel-level metrics (brightness, RMS contrast, center-vs-edge energy ratio for subject isolation/bokeh, and RGB means/standard deviations), which we combine with aspect ratio, width, and height.

In [ ]:
def trim_uniform_borders(img: Image.Image, tolerance: int = 15) -> Image.Image:
    """Trims artificial solid black/white contest borders from AVA images."""
    bg = Image.new(img.mode, img.size, img.getpixel((0, 0)))
    diff = ImageChops.difference(img, bg)
    diff = ImageChops.add(diff, diff, 2.0, -tolerance)
    bbox = diff.getbbox()
    return img.crop(bbox) if bbox else img


def extract_structural_features(img: Image.Image) -> list[float]:
    """Extracts lightweight color and structural descriptors for clustering & PCA."""
    arr = np.asarray(img.resize((128, 128)), dtype=np.float32) / 255.0
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

    # Luminance (perceived brightness) and RMS contrast
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    contrast = float(np.std(lum))
    brightness = float(np.mean(lum))

    # Center vs. global energy ratio (proxy for subject isolation / shallow depth of field)
    center_lum = float(np.std(lum[32:96, 32:96]))
    center_ratio = center_lum / (contrast + 1e-5)

    return [
        brightness,
        contrast,
        center_ratio,
        float(np.mean(r)), float(np.mean(g)), float(np.mean(b)),
        float(np.std(r)), float(np.std(g)), float(np.std(b)),
    ]

## Step 4: Stream, Clean, and Extract Features in Batches
We stream `trojblue/AVA-Huggingface` (or the 10% stratified sample `trojblue/AVA-aesthetics-10pct-min50-10bins` for fast prototyping) directly over HTTP so we never have to store 35 GB of raw JPEGs on disk.

**Cleaning rules applied:**
1. Force 3-channel RGB (`img.convert("RGB")`) to handle grayscale and 4-channel RGBA images.
2. Trim uniform contest borders.
3. Filter out corrupted images, tiny thumbnails ($< 128 \times 128$), and extreme banner aspect ratios ($> 4:1$ or $< 1:4$).
4. Verify at least 50 total votes and compute both the weighted **mean aesthetic score ($\mu$)** and **vote standard deviation ($\sigma$)**.

In [ ]:
# Tip: Use "trojblue/AVA-aesthetics-10pct-min50-10bins" (~25k images) for a fast 15-min run,
# or "trojblue/AVA-Huggingface" (~250k images) for the full dataset.
DATASET_NAME = "trojblue/AVA-aesthetics-10pct-min50-10bins"
BATCH_SIZE = 64
MAX_SAMPLES = 25000  # Set to None to process the entire dataset

stream_ds = load_dataset(DATASET_NAME, split="train", streaming=True)

cleaned_records = []
batch_imgs, batch_meta = [], []
skipped_count = 0

pbar = tqdm(total=MAX_SAMPLES, desc="Cleaning & Embedding AVA")

with torch.no_grad():
    for row in stream_ds:
        if MAX_SAMPLES and len(cleaned_records) >= MAX_SAMPLES:
            break
        try:
            img = row["image"]
            if img is None:
                skipped_count += 1
                continue
            img = img.convert("RGB")

            # 1. Trim artificial borders and check geometry
            img = trim_uniform_borders(img)
            w, h = img.size
            aspect_ratio = w / max(h, 1)
            if w < 128 or h < 128 or aspect_ratio > 4.0 or aspect_ratio < 0.25:
                skipped_count += 1
                continue

            # 2. Validate vote counts and compute mean & std score
            counts = np.array(row["rating_counts"], dtype=np.float64)
            total_votes = counts.sum()
            if total_votes < 50:
                skipped_count += 1
                continue

            scores = np.arange(1, 11, dtype=np.float64)
            probs = counts / total_votes
            mean_score = float(np.sum(probs * scores))
            std_score = float(np.sqrt(np.sum(probs * (scores - mean_score) ** 2)))

            # 3. Extract 12-dim structural feature vector
            struct_feats = extract_structural_features(img) + [aspect_ratio, float(w), float(h)]

            batch_imgs.append(img)
            batch_meta.append({
                "image_id": str(row["image_id"]),
                "mean_score": mean_score,
                "std_score": std_score,
                "total_votes": int(total_votes),
                "rating_counts": row["rating_counts"],
                "structural_features": struct_feats,
            })

            # 4. Run batch through frozen CLIP vision encoder
            if len(batch_imgs) == BATCH_SIZE:
                inputs = processor(images=batch_imgs, return_tensors="pt").to(device)
                vision_outputs = model.vision_model(pixel_values=inputs["pixel_values"])
                embeds = model.visual_projection(vision_outputs.pooler_output)
                embeds = torch.nn.functional.normalize(embeds, p=2, dim=-1)
                embeds = embeds / embeds.norm(p=2, dim=-1, keepdim=True)
                embeds_np = embeds.cpu().numpy().astype(np.float32)

                for meta, emb in zip(batch_meta, embeds_np):
                    meta["clip_embedding"] = emb.tolist()
                    cleaned_records.append(meta)

                pbar.update(len(batch_imgs))
                batch_imgs, batch_meta = [], []

        except Exception as exc:
            print(f"Processing failed: {type(exc).__name__}: {exc}")
            raise

    # Flush any remaining images in the final partial batch
    if batch_imgs:
        inputs = processor(images=batch_imgs, return_tensors="pt").to(device)
        embeds = model.get_image_features(**inputs)
        vision_outputs = model.vision_model(pixel_values=inputs["pixel_values"])
        embeds = model.visual_projection(vision_outputs.pooler_output)
        embeds = torch.nn.functional.normalize(embeds, p=2, dim=-1)
        embeds_np = embeds.cpu().numpy().astype(np.float32)

        for meta, emb in zip(batch_meta, embeds_np):
            meta["clip_embedding"] = emb.tolist()
            cleaned_records.append(meta)
        pbar.update(len(batch_imgs))

pbar.close()
print(f"Finished! Retained {len(cleaned_records)} clean images, filtered out {skipped_count} noisy/corrupted records.")

Cleaning & Embedding AVA:   5%|▌         | 1344/25000 [00:21<05:10, 76.19it/s]/Users/emilsvigants/miniforge3-dtu/envs/dtu02452/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Cleaning & Embedding AVA:  29%|██▉       | 7232/25000 [01:40<04:12, 70.49it/s]/Users/emilsvigants/miniforge3-dtu/envs/dtu02452/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
Cleaning & Embedding AVA:  82%|████████▏ | 20420/25000 [04:32<01:01, 75.07it/s]

Finished! Retained 20420 clean images, filtered out 17 noisy/corrupted records.


## Step 5: Save to Parquet & Share with Your Team on Hugging Face Hub
This exports the cleaned tabular dataset (without the bulky raw image bytes) to a single `.parquet` file that you can upload to GitHub/Google Drive, or push directly to a shared Hugging Face repository.

In [ ]:
from pathlib import Path

clean_ds = Dataset.from_list(cleaned_records)

parquet_path = Path(
    "/Users/emilsvigants/Projects/aesthetic-unsplash-image-analysis"
) / "data" / "ava_cleaned_features.parquet"

parquet_path.parent.mkdir(parents=True, exist_ok=True)
clean_ds.to_parquet(str(parquet_path))
print(f"Saved to {parquet_path}")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  3.67ba/s]

Saved to /Users/emilsvigants/Projects/aesthetic-unsplash-image-analysis/data/ava_cleaned_features.parquet


## Step 6: Teammate & Grader Quick-Loader
Once the `.parquet` file or Hugging Face repository is shared, your teammates (and the TAs grading your submission) can load the feature matrices ($X$) and target labels ($y$) in seconds without needing a GPU.

In [ ]:
# Load directly from local Parquet file (or replace with your Hugging Face repo ID)
loaded_ds = Dataset.from_parquet("ava_cleaned_features.parquet")

# Extract input feature matrices (X) and target aesthetic scores (y)
X_clip = np.array(loaded_ds["clip_embedding"], dtype=np.float32)          # Shape: (N, 512)
X_struct = np.array(loaded_ds["structural_features"], dtype=np.float32)  # Shape: (N, 12)
X_combined = np.hstack([X_clip, X_struct])                               # Shape: (N, 524)

y_mean = np.array(loaded_ds["mean_score"], dtype=np.float32)             # Shape: (N,)
y_std = np.array(loaded_ds["std_score"], dtype=np.float32)               # Shape: (N,)

print(f"X_clip shape:     {X_clip.shape}")
print(f"X_struct shape:   {X_struct.shape}")
print(f"X_combined shape: {X_combined.shape}")
print(f"y_mean shape:     {y_mean.shape}")

Generating train split: 20420 examples [00:00, 91014.54 examples/s]


X_clip shape:     (20420, 512)
X_struct shape:   (20420, 12)
X_combined shape: (20420, 524)
y_mean shape:     (20420,)
